# Logistic Regression

This notebook fits an exploratory logistic regression model for all-cause 30-day inpatient readmission using the synthetic final analysis dataset. The model is for portfolio demonstration only and is not a validated clinical prediction tool.

## Load Final Analysis Dataset

In [ ]:

from pathlib import Path
import sys
import duckdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analysis"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SQL_SCRIPTS = [
    "sql/01_profile_source_tables.sql",
    "sql/02_define_eligible_inpatient_encounters.sql",
    "sql/03_define_index_encounter.sql",
    "sql/04_define_postdischarge_utilization.sql",
    "sql/05_define_30_day_readmission.sql",
    "sql/06_create_final_analysis_dataset.sql",
]

def load_final_analysis_dataset():
    db_path = "/tmp/ehr_readmission_analysis_notebook.duckdb"
    con = duckdb.connect(db_path)
    for script in SQL_SCRIPTS:
        con.execute((PROJECT_ROOT / script).read_text())
    df = con.execute("SELECT * FROM final_analysis_dataset").fetchdf()
    con.close()
    return df

df = load_final_analysis_dataset()
print(f"Loaded final analysis dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")


## Prepare Modeling Variables

The model is intentionally parsimonious because the current synthetic cohort has a small number of 30-day readmission events. Outpatient follow-up is included as an exploratory covariate and should not be interpreted causally.

In [ ]:

import statsmodels.api as sm
import matplotlib.pyplot as plt

model_df = df.copy()
model_df["male_sex"] = (model_df["sex"] == "M").astype(int)
model_df["age_per_10_years"] = model_df["age_at_index"] / 10
model_df["log_length_of_stay"] = np.log1p(model_df["length_of_stay_days"])

predictors = [
    "age_per_10_years",
    "male_sex",
    "log_length_of_stay",
    "prior_encounters_12mo",
    "chronic_condition_count",
    "outpatient_followup_30d",
]

analysis_df = model_df[["readmitted_30d"] + predictors].dropna().copy()
print(f"Model rows: {len(analysis_df):,}")
print(f"Readmission events: {int(analysis_df['readmitted_30d'].sum()):,}")


## Fit Logistic Regression

In [ ]:

X = sm.add_constant(analysis_df[predictors].astype(float))
y = analysis_df["readmitted_30d"].astype(int)
model = sm.Logit(y, X).fit(disp=0, maxiter=200)

params = model.params
conf = model.conf_int()
results = pd.DataFrame({
    "term": params.index,
    "coefficient": params.values,
    "odds_ratio": np.exp(params.values),
    "ci_lower": np.exp(conf[0].values),
    "ci_upper": np.exp(conf[1].values),
    "p_value": model.pvalues.values,
})
term_labels = {
    "const": "Intercept",
    "age_per_10_years": "Age, per 10 years",
    "male_sex": "Male sex",
    "log_length_of_stay": "Log length of stay",
    "prior_encounters_12mo": "Prior encounters, 12 months",
    "chronic_condition_count": "Chronic condition count",
    "outpatient_followup_30d": "Outpatient follow-up within 30 days",
}
results["label"] = results["term"].map(term_labels).fillna(results["term"])
results = results[["term", "label", "coefficient", "odds_ratio", "ci_lower", "ci_upper", "p_value"]]
for col in ["coefficient", "odds_ratio", "ci_lower", "ci_upper", "p_value"]:
    results[col] = results[col].round(4)
results.to_csv(OUTPUT_DIR / "logistic_regression_results.csv", index=False)
results


## Model Summary Notes

In [ ]:

model_notes = pd.DataFrame([{
    "model_type": "Logistic regression",
    "outcome": "30-day inpatient readmission",
    "n_observations": int(model.nobs),
    "readmission_events": int(y.sum()),
    "converged": bool(model.mle_retvals.get("converged", False)),
    "pseudo_r_squared": round(float(model.prsquared), 4),
    "llr_p_value": round(float(model.llr_pvalue), 4),
    "interpretation_note": "Exploratory synthetic-data model; associations only; not a validated clinical prediction tool.",
}])
model_notes.to_csv(OUTPUT_DIR / "logistic_regression_model_notes.csv", index=False)
model_notes


## Odds Ratio Plot

In [ ]:

plot_df = results[results["term"] != "const"].copy().sort_values("odds_ratio")
fig, ax = plt.subplots(figsize=(8.5, 5.5))
y_pos = np.arange(len(plot_df))
ax.errorbar(
    plot_df["odds_ratio"],
    y_pos,
    xerr=[plot_df["odds_ratio"] - plot_df["ci_lower"], plot_df["ci_upper"] - plot_df["odds_ratio"]],
    fmt="o",
    color="#2f5f73",
    ecolor="#7a8a92",
    capsize=3,
)
ax.axvline(1, color="#8f3f2f", linestyle="--", linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["label"])
ax.set_xlabel("Odds ratio, log scale")
ax.set_xscale("log")
ax.set_title("Exploratory Logistic Regression for 30-Day Readmission")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "logistic_regression_odds_ratios.png", dpi=180)
plt.close(fig)
print((FIGURE_DIR / "logistic_regression_odds_ratios.png").relative_to(PROJECT_ROOT))


## Output Files

In [ ]:

for path in sorted(OUTPUT_DIR.glob("logistic_regression*.csv")):
    print(path.relative_to(PROJECT_ROOT))
